# Counterexample Commons — Complete Public Research Lab
### Exact finite validation, visual exploration, safe public demo, and optional private AI experiments

> **Runtime validation candidate**
>
> This notebook is being tested as a complete Google Colab workflow.
> Fresh Colab runtime validation is not yet complete.
>
> Scientific boundary:
> - Exact finite baseline calculations are locally reproducible.
> - AI-generated candidates are hypotheses until independently validated.
> - Sawin's explicit bound n^{1.014} is source-documented only and is not locally reproduced in this repository.

## Section 0 — Project Scope and Scientific Boundaries

### What is the Unit-Distance Problem?

Given n points in the plane, the unit-distance function u(n) counts the maximum number of pairs at exactly Euclidean distance 1. The Erdos Unit-Distance Problem asks how fast u(n) can grow.

### What did OpenAI (2026-05-20) claim?

That an AI-assisted construction produces a fixed delta > 0 such that u(n) > n^{1+delta} for infinitely many n. The companion paper by Alon et al. provides the mathematical argument.

### What does Sawin (arXiv:2605.20579) document?

An explicit construction achieving u(n) >= n^{1.014} for infinitely many n.

### What can this notebook actually verify?

```
SOURCE_DOCUMENTED (not locally reproduced):
  - OpenAI fixed-delta theorem
  - Sawin explicit bound n^1.014

LOCALLY_REPRODUCED_EXACT (finite only):
  - finite line baselines
  - finite square grid baselines
  - finite rational point configurations
  - rational mesh baseline (not Sawin construction)

NOT LOCALLY REPRODUCED:
  - Sawin algebraic-number-theoretic construction
  - asymptotic exponent n^1.014
```

No marketing language. This notebook does not prove a new mathematical bound.

## Section 1 — Fresh Colab Setup

This cell clones and installs the repository in Google Colab, or locates it locally for precheck runs.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = (
    "google.colab" in sys.modules
    or bool(os.environ.get("COLAB_RELEASE_TAG"))
    or bool(os.environ.get("COLAB_BACKEND_VERSION"))
)
REPO_URL = "https://github.com/error-wtf/counterexample-commons.git"
REPO_BRANCH = "rebuild/colab-complete-lab"
COLAB_REPO_DIR = Path("/content/counterexample-commons")

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
             REPO_URL, str(COLAB_REPO_DIR)], check=True,
        )
    else:
        subprocess.run(
            ["git", "-C", str(COLAB_REPO_DIR), "fetch", "origin", REPO_BRANCH],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(COLAB_REPO_DIR), "checkout", "-B",
             REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True,
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(COLAB_REPO_DIR)],
        check=True,
    )
    repo_root = COLAB_REPO_DIR
    runtime_label = "GOOGLE COLAB RUNTIME TEST"
else:
    start = Path.cwd().resolve()
    repo_root = next(
        (p for p in [start, *start.parents]
         if (p / "counterexample_commons").is_dir()
         and (p / "pyproject.toml").is_file()),
        None,
    )
    if repo_root is None:
        raise RuntimeError(
            "Repository root not found. Run from within a cloned "
            "counterexample-commons checkout or open in Colab."
        )
    runtime_label = "LOCAL PRECHECK ONLY — NOT GOOGLE COLAB EVIDENCE"

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import counterexample_commons

print(runtime_label)
print(f"Repository branch candidate: {REPO_BRANCH}")
print(f"Package version: {counterexample_commons.__version__}")


## Section 2 — Lab Self-Test

Verify the exact validator is operational before proceeding. Known unit-distance pair: (0,0) and (3/5, 4/5) — distance exactly 1.

In [ ]:
from counterexample_commons.validators import count_unit_edges_exact, validate_custom_configuration
from sympy import Rational

# Known unit-distance pair: (0,0) to (3/5, 4/5) has distance exactly 1
test_pts = [(Rational(0), Rational(0)), (Rational(3, 5), Rational(4, 5))]
count, edges = count_unit_edges_exact(test_pts)

assert count == 1, f"FAIL: expected 1 unit edge, got {count}"
assert edges == [(0, 1)], f"FAIL: unexpected edge list {edges}"

print("LAB_SELF_TEST: PASS")
print("Exact rational unit-distance validator is operational.")
print(f"(0,0) to (3/5, 4/5): distance^2 = (3/5)^2 + (4/5)^2 = 9/25 + 16/25 = 25/25 = 1 — confirmed exact.")


## Section 3 — Exact Finite Baselines

All calculations use SymPy exact rational arithmetic. Results are finite and exact — not asymptotic bounds.

> **Scope**: `LOCALLY_REPRODUCED_EXACT_FINITE_RESULT` only.  
> Sawin n^1.014 is `SOURCE_DOCUMENTED` and is not reproduced here.

In [ ]:
import sys
sys.path.insert(0, str(repo_root))
from counterexample_commons.validators import (
    validate_line_configuration,
    validate_grid_configuration,
    validate_custom_configuration,
)
from case_studies.erdos_unit_distance_2026.rational_mesh_baseline import verify_rational_mesh

results = []

# A. Line configurations
for n in [3, 5, 10, 20]:
    r = validate_line_configuration(n)
    results.append({
        "Configuration": f"Line n={n}",
        "Points": r["n"],
        "Exact unit edges": r["actual_edges"],
        "Expected": r["expected_edges"],
        "Status": "PASS" if r["pass"] else "FAIL",
        "Scope limitation": "Finite exact only",
    })

# B. Square grid configurations
for k in [3, 5, 8]:
    r = validate_grid_configuration(k)
    results.append({
        "Configuration": f"Grid {k}x{k}",
        "Points": r["n"],
        "Exact unit edges": r["actual_edges"],
        "Expected": r["expected_edges"],
        "Status": "PASS" if r["pass"] else "FAIL",
        "Scope limitation": "Finite exact only",
    })

# C. Rational non-axial configuration (3/5, 4/5 example)
custom_coords = [("0","0"),("1","0"),("0","1"),("3/5","4/5"),("3/5","-4/5")]
r = validate_custom_configuration(custom_coords)
results.append({
    "Configuration": "Rational non-axial (3/5, 4/5)",
    "Points": r["n"],
    "Exact unit edges": r["actual_edges"],
    "Expected": "N/A",
    "Status": r["status"],
    "Scope limitation": "Finite exact only",
})

# D. Rational mesh baseline
rm = verify_rational_mesh(5)
results.append({
    "Configuration": f"Rational mesh m=5",
    "Points": rm["n"],
    "Exact unit edges": rm["unit_edges"],
    "Expected": "N/A",
    "Status": rm["claim_status"],
    "Scope limitation": "Finite exact baseline only; not Sawin construction and not evidence for exponent 1.014.",
})

# Display table
print(f"{'Configuration':<35} {'Points':>8} {'Unit Edges':>12} {'Expected':>10} {'Status'}")
print("-" * 80)
for row in results:
    print(f"{row['Configuration']:<35} {str(row['Points']):>8} {str(row['Exact unit edges']):>12} {str(row['Expected']):>10}  {row['Status']}")
print()
print("Rational mesh scope note: Finite exact baseline only.")
print("Not Sawin s asymptotic construction and not evidence for exponent 1.014.")


## Section 4 — Visualisation of Unit-Distance Configurations

Plots below show exact validated unit-distance edges only. Axes are equal scale.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sympy import Rational
from counterexample_commons.validators import validate_grid_configuration, validate_custom_configuration

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

r = validate_grid_configuration(5)
pts = [(Rational(i % 5), Rational(i // 5)) for i in range(25)]
ax = axes[0]
xs = [float(q[0]) for q in pts]
ys = [float(q[1]) for q in pts]
for i, j in r["edges"]:
    ax.plot([xs[i], xs[j]], [ys[i], ys[j]], "b-", lw=0.8, alpha=0.5)
ax.scatter(xs, ys, s=30, c="navy", zorder=5)
ax.set_aspect("equal")
ax.set_title(
    "Exact finite grid baseline 5x5 ({} unit edges)\nnot Sawin construction"
    .format(r["actual_edges"])
)
ax.set_xlabel("x"); ax.set_ylabel("y")

coords = [("0","0"),("1","0"),("0","1"),("3/5","4/5"),("3/5","-4/5")]
r2 = validate_custom_configuration(coords)
pts2 = [(Rational(x), Rational(y)) for x, y in coords]
xs2 = [float(q[0]) for q in pts2]
ys2 = [float(q[1]) for q in pts2]
ax2 = axes[1]
for i, j in r2["edges"]:
    ax2.plot([xs2[i], xs2[j]], [ys2[i], ys2[j]], "r-", lw=1.2, alpha=0.7)
ax2.scatter(xs2, ys2, s=50, c="darkred", zorder=5)
for k, (x, y) in enumerate(zip(xs2, ys2)):
    ax2.annotate(coords[k], (x, y), textcoords="offset points", xytext=(5,5), fontsize=8)
ax2.set_aspect("equal")
ax2.set_title(
    "Rational non-axial config ({} unit edges)\nfinite exact only"
    .format(r2["actual_edges"])
)
ax2.set_xlabel("x"); ax2.set_ylabel("y")

plt.tight_layout()
plt.savefig("/tmp/lab_viz.png", dpi=120, bbox_inches="tight")
plt.show()
print("Visualisation generated. Axes equal scale. Edges drawn only for exact unit-distance pairs.")


## Section 5 — Interactive Configuration Explorer

Enter rational point coordinates (e.g. `3/5`) to compute exact unit-distance edges.

> Finite exact validation only. This explorer does not reproduce Sawin s asymptotic construction and does not generate formal proofs.

In [ ]:
import gradio as gr
from sympy import Rational
from counterexample_commons.validators import validate_custom_configuration, count_unit_edges_exact
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, io, base64

PRESETS = {
    "Unit Square": "0, 0\n1, 0\n1, 1\n0, 1",
    "Rational 3/5-4/5 Example": "0, 0\n1, 0\n0, 1\n3/5, 4/5\n3/5, -4/5",
    "Small Rational Mesh (m=2)": "0, 0\n1/2, 0\n1, 0\n0, 1/2\n1/2, 1/2\n1, 1/2\n0, 1\n1/2, 1\n1, 1",
}

def make_plot(coords, result):
    pts = [(Rational(x), Rational(y)) for x, y in coords]
    xs = [float(p[0]) for p in pts]
    ys = [float(p[1]) for p in pts]
    fig, ax = plt.subplots(figsize=(5, 5))
    for i, j in result["edges"]:
        ax.plot([xs[i], xs[j]], [ys[i], ys[j]], "b-", lw=1.2, alpha=0.6)
    ax.scatter(xs, ys, s=50, c="navy", zorder=5)
    ax.set_aspect("equal")
    ax.set_title(f"{result['n']} points, {result['actual_edges']} unit edges (exact)")
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    plt.close()
    buf.seek(0)
    return buf

def validate_points(text):
    if not text.strip():
        return "No input provided.", None
    try:
        coords = []
        for line in text.strip().splitlines():
            line = line.strip()
            if not line:
                continue
            parts = [p.strip() for p in line.split(",")]
            if len(parts) != 2:
                return f"Error: expected 2 values per line, got: {line!r}", None
            coords.append((parts[0], parts[1]))
        if len(coords) < 2:
            return "Need at least 2 points.", None
        result = validate_custom_configuration(coords)
        summary = (
            f"Points: {result['n']}\n"
            f"Exact unit-distance edges: {result['actual_edges']}\n"
            f"Status: {result['status']}\n"
            f"Edges (index pairs): {result['edges']}\n\n"
            "Finite exact validation only. Not Sawin construction."
        )
        buf = make_plot(coords, result)
        return summary, buf
    except Exception as e:
        return f"Error: {e}", None

with gr.Blocks(title="Unit-Distance Explorer") as explorer:
    gr.Markdown("### Interactive Exact Unit-Distance Explorer")
    gr.Markdown("> Finite exact validation only. No asymptotic bounds.")
    with gr.Row():
        with gr.Column():
            preset = gr.Dropdown(choices=list(PRESETS.keys()), label="Load preset")
            points_input = gr.Textbox(
                label="Points (one per line: x, y — rationals OK, e.g. 3/5)",
                lines=8, value="0, 0\n1, 0\n0, 1"
            )
            run_btn = gr.Button("Validate")
        with gr.Column():
            result_text = gr.Textbox(label="Result", lines=8)
            plot_out = gr.Image(label="Plot", type="pil")

    def load_preset(name):
        return PRESETS.get(name, "")

    preset.change(load_preset, inputs=preset, outputs=points_input)
    run_btn.click(validate_points, inputs=points_input, outputs=[result_text, plot_out])

print("Interactive explorer built. Call explorer.launch() to start.")
print("In Colab: the explorer will be embedded below when launched.")
if IN_COLAB:
    explorer.launch(share=False, inline=True)
else:
    print("Local precheck: explorer.launch() available interactively.")


## Section 6 — Read-Only Claim Registry

This registry is **read-only**. Claims cannot be upgraded by notebook execution. No status dropdowns. No auto-promotion.

In [ ]:
from counterexample_commons.claims import INITIAL_CLAIMS

print(f"{'Claim ID':<30} {'Status':<35} {'Locally Validated':<20} {'Limitations (truncated)'}")
print("-" * 120)
for c in INITIAL_CLAIMS:
    lim = c.limitations[:60] + "..." if len(c.limitations) > 60 else c.limitations
    print(f"{c.claim_id:<30} {c.status.value:<35} {str(c.locally_validated):<20} {lim}")

print()
print("REGISTRY_MODE: READ_ONLY")
print("No claim has been upgraded by this notebook run.")
print("UD-SAWIN-2026-001 status: SOURCE_DOCUMENTED, locally_validated=False — unchanged.")


## Section 7 — Safe Public Gradio Demo

**Mode: `colab-public-demo`** — no secrets, no provider calls, claim registry read-only.

Run this cell manually after verifying the sections above. A share link will appear in the output.

In [ ]:
PUBLIC_DEMO_ALLOWED = not globals().get("ENABLE_PRIVATE_AI_LAB", False)

if not PUBLIC_DEMO_ALLOWED:
    raise RuntimeError(
        "Cannot start public share demo while ENABLE_PRIVATE_AI_LAB is True. "
        "Disable private AI lab before launching public demo."
    )

from app.main import build_app
from counterexample_commons.config import AppMode
import os

os.environ["COUNTEREXAMPLE_MODE"] = "colab-public-demo"
demo = build_app(AppMode.COLAB_PUBLIC_DEMO)

print("Launching safe public demo.")
print("Mode: colab-public-demo")
print("No secrets. No live providers. Claim registry read-only.")
print()
if IN_COLAB:
    demo.queue().launch(share=True, debug=False)
else:
    print("LOCAL PRECHECK: Gradio launch skipped (not IN_COLAB).")
    print("Run demo.queue().launch() interactively if needed.")


## Section 8 — Export Validated Finite Result

Generates a JSON report of the finite baseline results. No asymptotic claims.

In [ ]:
import json
from datetime import datetime, timezone
from counterexample_commons.validators import validate_grid_configuration
from counterexample_commons.experiments.sanitization import sanitize_for_export, is_safe_for_export

result = validate_grid_configuration(5)
report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "runtime": runtime_label,
    "configuration": result["configuration"],
    "n_points": result["n"],
    "exact_unit_edges": result["actual_edges"],
    "expected_edges": result["expected_edges"],
    "pass": result["pass"],
    "scope": "LOCALLY_REPRODUCED_EXACT_FINITE_RESULT",
    "sawin_status": "SOURCE_DOCUMENTED — not locally reproduced",
    "scope_limitation": (
        "Finite exact validation export only; not an asymptotic theorem, "
        "not a Sawin reproduction, and not a formal proof of a new bound."
    ),
}
export_text = json.dumps(report, indent=2)
safe, reason = is_safe_for_export(export_text)
print(f"Safe for export: {safe} ({reason})")

sanitized = sanitize_for_export(export_text)
export_path = "/tmp/counterexample_commons_lab_export.json"
with open(export_path, "w") as f:
    f.write(sanitized)

print(f"Export written to: {export_path}")
print()
print(sanitized)

if IN_COLAB:
    try:
        from google.colab import files
        files.download(export_path)
        print("Download initiated.")
    except Exception as e:
        print(f"Download via files.download() unavailable: {e}")
        print(f"Manually download from: {export_path}")


## Section 9 — Optional Private AI Candidate Lab — Disabled by Default

This section is optional and private.
It is **disabled by default**.

> Do not use public share mode while private provider credentials are loaded.
> AI-generated candidates remain hypotheses until independently validated by the exact finite validator.

In [ ]:
ENABLE_PRIVATE_AI_LAB = False

if not ENABLE_PRIVATE_AI_LAB:
    print("Private AI Lab disabled.")
    print("Set ENABLE_PRIVATE_AI_LAB = True only in a private session with keys loaded via Colab Secrets.")
    print("Do not enable this in a public share session.")
else:
    # Safety: block public demo after enabling private lab
    PUBLIC_DEMO_ALLOWED = False

    print("WARNING: Private AI Lab enabled.")
    print("PUBLIC_DEMO_ALLOWED has been set to False.")
    print("Do NOT start a public Gradio share link in this session.")
    print()
    print("To use AI candidate generation:")
    print("  1. Load your API key via Colab Secrets (not in this cell).")
    print("  2. Set COUNTEREXAMPLE_MODE to colab-private.")
    print("  3. Use the experiment pipeline from counterexample_commons.experiments")
    print()
    print("STATUS: PRIVATE_OPTIONAL_WORKFLOW — not publicly validated, not a live demo.")
    print("No API calls are made in this cell. Manual activation required.")


## Section 10 — Validation Status and Limitations

This notebook distinguishes between:

- exact finite computations performed in this runtime;
- published source-documented mathematical claims;
- optional AI-generated hypotheses;
- results not reproduced locally.

**Current boundaries:**

- Finite line/grid/rational configurations can be exactly validated here.
- The finite rational mesh baseline is not Sawin's construction.
- Sawin's exponent n^{1.014} is source-documented only.
- A successful public UI launch proves interface availability, not a new mathematical theorem.
- AI-provider experiments, if manually enabled, remain hypotheses until exact validation.

### Runtime Checklist

| Check | Status |
|-------|--------|
| Fresh Colab bootstrap | user verifies after real run |
| Exact finite self-test | produced during runtime |
| Baseline table | produced during runtime |
| Visualisations | produced during runtime |
| Explorer interaction | manually tested |
| Public Gradio demo | manually tested |
| Export | manually tested |
| Private AI lab | disabled by default / optional |
